# Issue #13 [Data/Viz]: Reporte Visual Accesible, Control de Sesgos y Storytelling para YEB

**Subentorno:** `analytics/` | **Rama:** `feat/JMCM-analytics-nosql` | **Script equivalente:** `scripts/13_visualizacion_storytelling_yeb.py`  
**Criterios de Aceptación:** `CA-ISS-13-01` a `CA-ISS-13-04` | **Riesgos mitigados:** `NEG-011`, `UX-002` | **Casos de prueba:** `CP-053`, `CP-053B`, `CP-053C`

Este cuaderno transforma los indicadores validados en las Semanas 5 y 6 en tres visualizaciones comprensibles y accesibles (Matplotlib `.png`, Seaborn `stripplot` `.png` y Plotly interactivo `.html` + `.csv`), documenta la corrección antes/después de un sesgo visual por eje truncado y presenta el microinforme de storytelling en dos voces (técnica y ejecutiva).

## 1. Validación previa con `assert` del universo analizable (`CA-ISS-13-01` / `NEG-011`)

- **Universo analizable (`n = 93`):** postulaciones con `origen_registro == 'dataset_preparado'` al corte `2026-02-02`.
- **Exclusiones visibles:** `4` registros en `cuarentena_issue_05` (`P1 - Revisar identidad o edad`), y `38` postulaciones activas (`NaN`) cuando se grafica el subconjunto de tiempos de respuesta (`count = 55`).

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
if (cwd / 'data' / 'processed').exists():
    ANALYTICS_ROOT = cwd
elif (cwd / 'analytics' / 'data' / 'processed').exists():
    ANALYTICS_ROOT = cwd / 'analytics'
else:
    ANALYTICS_ROOT = cwd.parent

sys.path.insert(0, str(ANALYTICS_ROOT / 'scripts'))
import importlib
m13 = importlib.import_module('13_visualizacion_storytelling_yeb')

df_total, df_u, FECHA_CORTE = m13.cargar_datos_validados(ANALYTICS_ROOT)
OUTPUTS_DIR = ANALYTICS_ROOT / 'outputs'

assert int(df_u['status'].value_counts().sum()) == 93
assert int(df_u['modality'].value_counts().sum()) == 93
assert int(df_u['ciudad_normalizada'].value_counts().sum()) == 93
assert int(df_u['tiempo_respuesta_dias'].count()) == 55
print(f'Universo analizable verificado: n = {len(df_u)} | Respondidas: count = {int(df_u["tiempo_respuesta_dias"].count())} | Corte: {FECHA_CORTE} | Excluidas P1: {len(df_total) - len(df_u)}')

Universo analizable verificado: n = 93 | Respondidas: count = 55 | Corte: 2026-02-02 | Excluidas P1: 4


## 2. Figura 1 (`Matplotlib .png`): Barras horizontales de embudo y prioridad con eje desde cero y `bar_label` (`CA-ISS-13-02`)

- **Texto alternativo (`alt text`):** *Gráfico de barras horizontales desde cero al corte 2026-02-02. En el Panel A (n=93 postulaciones válidas, 4 excluidas en cuarentena), el embudo registra: Enviada (submitted) 20 (21,51%), En revisión (under_review) 18 (19,35%), Entrevista (interview) 17 (18,28%), Contratada (accepted) 20 (21,51%) y Rechazada (rejected) 18 (19,35%). En el Panel B (n=97 recibidas), la cola de prioridad muestra: P1 Revisar identidad o edad 4 (4,12%), P2 Postulación desactualizada 34 (35,05%), P3 Apto para entrevista 1 (1,03%) y Sin acción prioritaria 58 (59,79%).*

In [2]:
png1, csv1 = m13.generar_fig01_matplotlib(df_total, df_u, FECHA_CORTE, OUTPUTS_DIR)
respaldo_fig1 = pd.read_csv(csv1)
assert int(respaldo_fig1.loc[respaldo_fig1['panel'].str.startswith('A'), 'conteo'].sum()) == 93
assert int(respaldo_fig1.loc[respaldo_fig1['panel'].str.startswith('B'), 'conteo'].sum()) == 97
print(f'Exportados: {png1.relative_to(ANALYTICS_ROOT)} y {csv1.relative_to(ANALYTICS_ROOT)}')
respaldo_fig1

Exportados: outputs\fig01_embudo_postulaciones_matplotlib.png y outputs\fig01_embudo_postulaciones_respaldo.csv


,panel,categoria,conteo,denominador_n,porcentaje,fecha_corte,exclusiones
0,A - Estado del embudo (universo analizable),1. Enviada (submitted),20,93,21.51,2026-02-02,4 registros de cuarentena_issue_05 (P1)
1,A - Estado del embudo (universo analizable),2. En revisión (under_review),18,93,19.35,2026-02-02,4 registros de cuarentena_issue_05 (P1)
2,A - Estado del embudo (universo analizable),3. Entrevista (interview),17,93,18.28,2026-02-02,4 registros de cuarentena_issue_05 (P1)
3,A - Estado del embudo (universo analizable),4. Contratada (accepted),20,93,21.51,2026-02-02,4 registros de cuarentena_issue_05 (P1)
4,A - Estado del embudo (universo analizable),5. Rechazada (rejected),18,93,19.35,2026-02-02,4 registros de cuarentena_issue_05 (P1)
5,B - Cola de prioridad operativa (total recibido),P1 - Revisar identidad o edad,4,97,4.12,2026-02-02,0 (incluye las 4 filas en cuarentena como P1)
6,B - Cola de prioridad operativa (total recibido),P2 - Postulación desactualizada,34,97,35.05,2026-02-02,0 (incluye las 4 filas en cuarentena como P1)
7,B - Cola de prioridad operativa (total recibido),P3 - Apto para entrevista,1,97,1.03,2026-02-02,0 (incluye las 4 filas en cuarentena como P1)
8,B - Cola de prioridad operativa (total recibido),Sin acción prioritaria,58,97,59.79,2026-02-02,0 (incluye las 4 filas en cuarentena como P1)


## 3. Figura 2 (`Seaborn .png`): `stripplot` de tiempos de respuesta (`n=55`) y aspiración salarial (`n=93`) (`CA-ISS-13-02`)

Se emplea `stripplot` en lugar de *boxplot* para mostrar cada observación individual y evitar ocultar que la cohorte de menores protegidos respondidos tiene solo `12` observaciones (`3` en Híbrido, `4` en Remoto y `5` en Presencial).

- **Texto alternativo (`alt text`):** *Gráfico de puntos (stripplot) de Seaborn al corte 2026-02-02. El Panel A muestra cada una de las 55 postulaciones respondidas (38 activas en NaN y 4 en cuarentena excluidas): la mediana de tiempo de respuesta es 4,0 días en Remoto (n=18) e Híbrido (n=19), y 3,0 días en Presencial (n=18); los menores protegidos suman 12 observaciones con mediana global de 5,5 días frente a 3,0 días en jóvenes adultos (n=43). El Panel B muestra las 93 aspiraciones salariales válidas: medianas de 2,15 millones COP en Híbrido (n=31), 1,94 millones COP en Remoto (n=32) y 1,86 millones COP en Presencial (n=30).*

In [3]:
png2, csv2 = m13.generar_fig02_seaborn_stripplot(df_u, FECHA_CORTE, OUTPUTS_DIR)
respaldo_fig2 = pd.read_csv(csv2)
assert len(respaldo_fig2) == 93 and int(respaldo_fig2['tiempo_respuesta_dias'].count()) == 55
print(f'Exportados: {png2.relative_to(ANALYTICS_ROOT)} y {csv2.relative_to(ANALYTICS_ROOT)}')
respaldo_fig2.groupby(['modality', 'grupo_etario']).agg(
    n_size=('application_id', 'size'),
    n_respondidas=('tiempo_respuesta_dias', 'count'),
    mediana_dias=('tiempo_respuesta_dias', 'median'),
    mediana_salario_millones=('salario_millones_cop', 'median'),
).round(2)

Exportados: outputs\fig02_distribucion_tiempos_stripplot_seaborn.png y outputs\fig02_distribucion_tiempos_respaldo.csv


n_size  n_respondidas  mediana_dias  \
modality   grupo_etario                                                   
Híbrido    Joven adulto (18-24)         24             15           3.0   
           Menor protegido (16-17)       7              4           5.5   
Presencial Joven adulto (18-24)         21             12           3.0   
           Menor protegido (16-17)       9              6           4.5   
Remoto     Joven adulto (18-24)         27             16           4.0   
           Menor protegido (16-17)       5              2           5.5   

                                    mediana_salario_millones  
modality   grupo_etario                                       
Híbrido    Joven adulto (18-24)                         2.04  
           Menor protegido (16-17)                      2.19  
Presencial Joven adulto (18-24)                         1.84  
           Menor protegido (16-17)                      1.87  
Remoto     Joven adulto (18-24)                         1.94  
           Menor protegido (16-17)                      2.02

## 4. Figura 3 (`Plotly .html + .csv`): Indicadores por ciudad con `hovertemplate` personalizado y tabla CSV accesible (`CA-ISS-13-02` / `UX-002`)

- **Texto alternativo (`alt text`):** *Gráfico interactivo de barras agrupadas por ciudad al corte 2026-02-02 sobre n=93 postulaciones válidas (4 en cuarentena excluidas), respaldado por fig03_indicadores_ciudad_accesible.csv. Postulaciones totales y avance a entrevista/contratación por ciudad: Bogotá 17 postulaciones y 7 en entrevista/contratadas (41,18%, salario mediano 2.151.971 COP); Pereira 16 y 7 (43,75%, 1.883.850 COP); Cali 16 y 7 (43,75%, 2.025.630,50 COP); Medellín 15 y 7 (46,67%, 1.869.998 COP); Barranquilla 15 y 5 (33,33%, 1.794.093 COP); Bucaramanga 14 y 4 (28,57%, 2.214.090 COP).*

In [4]:
html3, csv3 = m13.generar_fig03_plotly_interactivo(df_u, FECHA_CORTE, OUTPUTS_DIR)
tabla_accesible_fig3 = pd.read_csv(csv3)
assert int(tabla_accesible_fig3['n_postulaciones_size'].sum()) == 93
assert int(tabla_accesible_fig3['n_respondidas_count'].sum()) == 55
print(f'Exportados: {html3.relative_to(ANALYTICS_ROOT)} y {csv3.relative_to(ANALYTICS_ROOT)}')
tabla_accesible_fig3

Exportados: outputs\fig03_indicadores_ciudad_plotly.html y outputs\fig03_indicadores_ciudad_accesible.csv


,ciudad_normalizada,departamento,n_postulaciones_size,n_candidatos_nunique,n_respondidas_count,n_entrevista_o_contratado,n_contratados,tasa_entrevista_pct,tasa_contratacion_pct,tiempo_resp_media_dias,tiempo_resp_mediana_dias,salario_esp_mediana_cop,modalidad_predominante,fecha_corte,exclusiones
0,Bogotá,Bogotá D.C.,17,17,11,7,3,41.18,17.65,4.55,4.0,1972608.0,Híbrido,2026-02-02,4 registros de cuarentena_issue_05 (P1)
1,Cali,Valle del Cauca,16,16,10,8,4,50.00,25.00,2.80,2.5,2023486.0,Remoto,2026-02-02,4 registros de cuarentena_issue_05 (P1)
2,Pereira,Risaralda,16,16,9,6,2,37.50,12.50,4.56,5.0,1843937.5,Remoto,2026-02-02,4 registros de cuarentena_issue_05 (P1)
3,Barranquilla,Atlántico,15,15,10,6,4,40.00,26.67,4.30,5.0,2088823.0,Presencial,2026-02-02,4 registros de cuarentena_issue_05 (P1)
4,Medellín,Antioquia,15,15,8,6,4,40.00,26.67,4.00,3.0,1843002.0,Presencial,2026-02-02,4 registros de cuarentena_issue_05 (P1)
5,Bucaramanga,Santander,14,14,7,4,3,28.57,21.43,3.71,3.0,2233809.5,Híbrido,2026-02-02,4 registros de cuarentena_issue_05 (P1)


## 5. Comparación Antes/Después de un Sesgo Visual (`CA-ISS-13-03` / `CP-053B`)

Se compara una gráfica con eje Y truncado (`38,0%` a `41,0%`) y dependencia exclusiva de colores semáforo frente a la versión corregida con eje completo desde `0%`, etiquetas directas `bar_label`, tramas geométricas `hatch` y contexto metodológico (`n=93`, corte `2026-02-02`, `4` exclusiones).

In [5]:
png4 = m13.generar_fig04_comparativa_sesgo(df_u, FECHA_CORTE, OUTPUTS_DIR)
assert png4.exists() and png4.stat().st_size > 0
print(f'Comparativa antes/después exportada: {png4.relative_to(ANALYTICS_ROOT)} ({png4.stat().st_size:,} bytes)')

Comparativa antes/después exportada: outputs\fig04_comparativa_sesgo_eje_truncado.png (162,268 bytes)


## 6. Microinforme de Storytelling en Dos Voces (200–300 palabras): Hallazgo + Evidencia + Limitación + Acción propuesta (`CA-ISS-13-04` / `CP-053C`)

**Voz 1 — Lectura técnica (Equipo de Datos e Ingeniería):**  
- **Hallazgo:** Al corte `2026-02-02`, el universo analizable (`n=93`, excluidos `4` registros en cuarentena `P1`) presenta una conversión global a entrevista o contratación del `39,78%` (`37/93`), pero concentra un cuello de botella operativo en las postulaciones activas: `34` de las `38` solicitudes en `submitted` o `under_review` (`89,47%`) superan el umbral del percentil 90 de respuesta (`7` días) y caen en `P2 - Postulación desactualizada`, mientras las `55` respondidas mantienen una mediana de `4,0` días (`IQR` acotado entre `2` y `6` días).  
- **Evidencia:** `FIG-YEB-01` confirma la cuadratura del embudo (`20+18+17+20+18=93`) y `FIG-YEB-02` (`stripplot`) muestra que en menores protegidos (`16–17` años) solo existen `12` casos respondidos con mediana de `5,5` días frente a `3,0` días en adultos jóvenes (`n=43`).  
- **Limitación:** Los conteos por subgrupo ciudad-modalidad incluyen `9` celdas vacías (`NaN`) y `3` celdas unitarias (`n=1`), por lo que las diferencias observadas son descriptivas y no prueban relaciones causales.  
- **Acción propuesta:** Integrar en `Modules/Dashboard` una alerta preventiva cuando `dias_sin_actualizar > 7` en estados activos y exigir revisión humana de los `4` casos `P1`.

**Voz 2 — Explicación ejecutiva no técnica (Administrador del Portal YEB):**  
- **Hallazgo y Evidencia:** Cuatro de cada diez postulaciones válidas (`39,78%`) logran entrevista o contratación y, cuando las empresas responden, suelen hacerlo en unos `4` días; sin embargo, `34` jóvenes que siguen esperando respuesta llevan más de una semana sin movimiento.  
- **Limitación:** Estos números son una fotografía del periodo y no indican qué empresa o ciudad sea mejor, pues varias combinaciones tienen muy pocos casos.  
- **Acción propuesta:** Priorizar hoy el recordatorio a empresas con esas `34` postulaciones estancadas y revisar manualmente los documentos legales de menores y los `4` perfiles en cuarentena antes de autorizar cualquier entrevista.

In [6]:
rutas_fichas = m13.verificar_fichas_y_storytelling(ANALYTICS_ROOT)
n_palabras = len(m13.MICROINFORME_STORYTELLING.split())
assert 200 <= n_palabras <= 300
print(f'Fichas verificadas: {len(rutas_fichas)} | Longitud microinforme: {n_palabras} palabras (rango exigido: 200-300).')

Fichas verificadas: 2 | Longitud microinforme: 286 palabras (rango exigido: 200-300).
